# Практическая работа № 1 — ЭТАЛОННОЕ РЕШЕНИЕ ПРЕПОДАВАТЕЛЯ
## Анализ цифрового образовательного следа средствами Google Colab и BigQuery

**Базовый вариант:** № 1, датасет OULAD, курс `BBB`, поток `2013J`.

**Аудитория:** студенты педагогических специальностей, учителя, методисты.

---

### Как пользоваться этим файлом

После каждой ячейки идёт блок **«Методический комментарий»** — что именно проговорить с аудиторией,
на какую типовую ошибку обратить внимание и как связать шаг с теорией (5V, классификация следа, этика).

### Тайминг занятия (пара 2 × 45 мин)

| Время | Блок | Содержание |
|---|---|---|
| 0–15 | Теория | 5V в образовании, виды цифрового следа, минимизация данных |
| 15–35 | Ячейки 1–2 | Chunking, EDA, гранулярность данных |
| 35–50 | Ячейки 3–4 | Ранние признаки, правило-сигнал, график |
| 50–70 | Ячейка 5 | BigQuery Sandbox: проект, датасет, загрузка, SQL |
| 70–85 | Ячейка 6 | Сверка Python и SQL |
| 85–90 | Ячейка 7 | Педагогический вывод и этика, выдача вариантов |

### Критерии оценивания (10 баллов)

| Блок | Баллы | Что считается выполненным |
|---|:---:|---|
| EDA и качество данных | 2 | Типы, пропуски, уникальные студенты, диапазон дней, вывод о шуме |
| SQL и агрегаты | 2 | Корректный `GROUP BY` с фильтром по периоду, осмысленные метрики |
| Оконная функция | 2 | Функция из своего варианта, верные `PARTITION BY` / `ORDER BY` / рамка |
| Правило / сигнал | 2 | Явные пороги, сравнение групп, отсутствие «чёрного ящика» |
| Педагогический вывод и этика | 2 | Альтернативные объяснения, корректные формулировки, конкретное действие |

**Автоматический незачёт блока «этика»:** формулировки вида «модель выявила слабого/ленивого ученика»,
включение демографии в правило, публикация списков обучающихся.

---
## Ячейка 1. Загрузка и нарезка реального массива данных OULAD

Архив UCI — 44,6 МБ, лицензия CC BY 4.0. Если доступ к `archive.ics.uci.edu` с локального IP закрыт,
преподаватель скачивает архив заранее и раздаёт его с корпоративного диска — код ниже поддерживает оба пути.

In [ ]:
import io
import zipfile
from pathlib import Path

import pandas as pd
import requests

URL = ("https://archive.ics.uci.edu/static/public/349/"
       "open%2Buniversity%2Blearning%2Banalytics%2Bdataset.zip")
LOCAL_ZIP = Path("open+university+learning+analytics+dataset.zip")          # запасной путь: архив, выданный преподавателем
RAW_DIR = Path("oulad_raw")

if RAW_DIR.exists() and (RAW_DIR / "studentVle.csv").exists():
    print("Архив уже распакован, скачивание пропущено.")
elif LOCAL_ZIP.exists():
    with zipfile.ZipFile(LOCAL_ZIP) as zf:
        zf.extractall(RAW_DIR)
    print("Распакован локальный архив:", LOCAL_ZIP)
else:
    r = requests.get(URL, timeout=180)
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
        zf.extractall(RAW_DIR)
    print("Архив скачан с UCI и распакован.")

print("Файлы датасета:", sorted(p.name for p in RAW_DIR.glob("*.csv")))

In [ ]:
# Читаем самый большой файл studentVle.csv (~10,6 млн записей) частями,
# чтобы не загружать весь массив в оперативную память Colab.
#
# Важно: одна строка studentVle — не «один клик».
# Это агрегированная запись для сочетания студент × день × ресурс,
# а sum_click хранит число взаимодействий с этим ресурсом в данный день.
CODE_MODULE, CODE_PRESENTATION = "BBB", "2013J"
MAX_DAY = 59

chunks = []
for chunk in pd.read_csv(RAW_DIR / "studentVle.csv", chunksize=400_000):
    subset = chunk[
        (chunk["code_module"] == CODE_MODULE)
        & (chunk["code_presentation"] == CODE_PRESENTATION)
        & (chunk["date"].between(0, MAX_DAY))
    ]
    if len(subset):
        chunks.append(subset)

events = pd.concat(chunks, ignore_index=True)
print(f"Строк взаимодействий после фильтрации: {len(events):,}")

# Контекст ресурсов и итоговый результат обучения
vle = pd.read_csv(RAW_DIR / "vle.csv")[
    ["code_module", "code_presentation", "id_site", "activity_type"]
]
students = pd.read_csv(RAW_DIR / "studentInfo.csv")[
    ["code_module", "code_presentation", "id_student", "final_result"]
]

practice = (
    events
    .merge(vle, on=["code_module", "code_presentation", "id_site"], how="left")
    .merge(students, on=["code_module", "code_presentation", "id_student"], how="left")
    .rename(columns={"date": "day"})
)

# Принцип минимизации данных: оставляем только поля, необходимые для учебной задачи
practice = practice[[
    "code_module", "code_presentation", "id_student",
    "day", "id_site", "sum_click", "activity_type", "final_result",
]]

practice.to_csv("practice01_oulad.csv", index=False)
size_mb = Path("practice01_oulad.csv").stat().st_size / 1024**2
print(f"Файл создан: строк {len(practice):,}, размер {size_mb:.2f} МБ")
assert size_mb < 90, "Срез слишком большой для учебной веб-загрузки в BigQuery"


> **Методический комментарий.**
>
> - **Volume.** Полный `studentVle.csv` содержит около 10,6 млн записей, поэтому мы читаем его частями.
>   После отбора курса `BBB`, потока `2013J` и дней `0–59` осталось **161 160 строк**.
> - **Полученный учебный файл имеет размер 6,66 МБ.** Это фактический результат данного среза,
>   а не ожидаемые 30–80 МБ. Для первой практической это даже удобнее: файл быстро загружается
>   и в Colab, и через веб-интерфейс BigQuery.
> - **Variety.** Смысл появляется после объединения трёх сущностей: журнала взаимодействий,
>   справочника ресурсов и итогового результата студента. В `pandas` используется `merge`,
>   в SQL аналогичная операция называется `JOIN`.
> - **Минимизация данных.** Мы сознательно не переносим в учебный файл `gender`, `region`,
>   `highest_education`, `imd_band`, `age_band`, `disability`, потому что они не нужны
>   для первого знакомства с анализом активности.
> - **Отрицательные даты.** В исходном OULAD встречаются значения `date < 0` — это действия
>   до официального начала курса. Для единого учебного окна оставляем только `0–59`.
> - Ассерт `< 90 МБ` здесь служит страховкой на случай, если преподаватель позднее расширит
>   период или выберет другой поток.


---
## Ячейка 2. Исследовательский анализ (EDA) и качество данных

In [ ]:
practice = pd.read_csv("practice01_oulad.csv")

print("Размерность:", practice.shape)
print("Уникальных обучающихся:", practice["id_student"].nunique())
print("Диапазон дней:", practice["day"].min(), "—", practice["day"].max())

print("\nТипы данных:")
print(practice.dtypes)

print("\nПропуски по столбцам:")
print(practice.isna().sum())

print("\nРаспределение итоговых результатов курса (на уровне человека):")
print(practice.groupby("id_student")["final_result"].first().value_counts())

print("\nРаспределение дневной активности (проверка экстремальных значений):")
daily = practice.groupby(["id_student", "day"])["sum_click"].sum()
print(daily.describe(percentiles=[0.5, 0.9, 0.99]))

> **Методический комментарий.**
>
> - **Гранулярность данных.** Одна строка учебной таблицы соответствует сочетанию
>   *обучающийся × день × ресурс*, а `sum_click` показывает число взаимодействий с ресурсом.
>   `final_result`, напротив, относится к обучающемуся в целом. Поэтому сначала нужно понять
>   уровень детализации, а уже затем агрегировать.
> - В срезе получено **161 160 строк**, **1 819 обучающихся**, диапазон дней **0–59**.
> - **Пропусков нет ни в одном из восьми полей.** Поэтому в этом конкретном срезе
>   `left merge` не создал пустых `activity_type`. В другом курсе или срезе это нужно проверять заново.
> - Итоговые результаты на уровне обучающегося: `Pass` — **889**, `Fail` — **458**,
>   `Withdrawn` — **296**, `Distinction` — **176**.
> - **Veracity.** Медиана дневной активности равна **10** взаимодействиям, 90-й перцентиль — **39**,
>   99-й — **115**, максимум — **444**. Экстремальное значение ещё не означает ошибку:
>   оно может отражать интенсивную работу, особенности навигации или специфику логирования.
> - По этим данным нельзя напрямую делать выводы о мотивации, способностях или добросовестности.
>   Лог показывает взаимодействия с платформой, а не все формы учебной деятельности.


---
## Ячейка 3. Ранние признаки (дни 0–29) и простое правило-сигнал

In [ ]:
EARLY_WINDOW = 30

early_activity = practice[practice["day"] < EARLY_WINDOW]

student_metrics = (
    early_activity
    .groupby("id_student")
    .agg(
        total_clicks=("sum_click", "sum"),
        active_days=("day", "nunique"),
        resources_used=("id_site", "nunique"),
        final_result=("final_result", "first"),
    )
    .reset_index()
)

# Целевой маркер успешности курса
student_metrics["is_success"] = (
    student_metrics["final_result"].isin(["Pass", "Distinction"]).astype(int)
)

# Пороги — нижний квартиль по обеим метрикам
q25_clicks = student_metrics["total_clicks"].quantile(0.25)
q25_days = student_metrics["active_days"].quantile(0.25)

student_metrics["support_signal"] = (
    (student_metrics["total_clicks"] <= q25_clicks)
    & (student_metrics["active_days"] <= q25_days)
)

print(f"Порог кликов Q25: {q25_clicks:.0f} | порог активных дней Q25: {q25_days:.0f}")
print(f"Сигнал получен у {student_metrics['support_signal'].sum()} из {len(student_metrics)} обучающихся")

summary = (
    student_metrics
    .groupby("support_signal")["is_success"]
    .agg(students_count="count", success_rate="mean")
    .round(3)
)
summary

> **Методический комментарий.**
>
> - Для первых 30 дней в расчёт попали **1 767 обучающихся**. В исходном 60-дневном срезе их 1 819:
>   **52 обучающихся не имели записей в дни 0–29**, поэтому не попали в раннюю агрегацию.
> - Полученные пороги нижнего квартиля: **34 клика** и **4 активных дня**.
>   Условию одновременно удовлетворяют **387 из 1 767** обучающихся.
> - В группе без сигнала доля `Pass/Distinction` равна **0,653 (65,3%)**,
>   в группе с сигналом — **0,364 (36,4%)**.
> - Это **ассоциация**, а не доказательство причинности. Нельзя утверждать,
>   что низкая активность *вызвала* более низкий результат.
> - Q25 выбран здесь как **простое учебное правило**, а не как «оптимальный» порог.
>   В реальной системе порог согласуют с педагогической задачей, ресурсом поддержки
>   и ценой ложных срабатываний.
> - Полезный эксперимент на вебинаре: заменить `0.25` на `0.10` и посмотреть,
>   как меняются размер группы сигнала и доля успешного завершения.


---
## Ячейка 4. Визуализация

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Слева — распределение ранней активности и порог
axes[0].hist(student_metrics["total_clicks"], bins=40, color="#4C78A8", edgecolor="white")
axes[0].axvline(q25_clicks, color="#E45756", linestyle="--", linewidth=2,
                label=f"Порог Q25 = {q25_clicks:.0f}")
axes[0].set_title("Распределение кликов за первые 30 дней", fontsize=12)
axes[0].set_xlabel("Суммарное число кликов")
axes[0].set_ylabel("Количество обучающихся")
axes[0].legend()

# Справа — доля успешного завершения курса по группам
rates = student_metrics.groupby("support_signal")["is_success"].mean()
labels = ["Без сигнала", "Сигнал: низкая активность"]
axes[1].bar(labels, [rates.get(False, 0), rates.get(True, 0)],
            color=["#4C78A8", "#E45756"], width=0.55)
for i, v in enumerate([rates.get(False, 0), rates.get(True, 0)]):
    axes[1].text(i, v + 0.02, f"{v:.0%}", ha="center", fontsize=11)
axes[1].set_ylim(0, 1)
axes[1].set_title("Доля успешного завершения курса", fontsize=12)
axes[1].set_ylabel("Доля Pass / Distinction")
axes[1].grid(axis="y", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

> **Методический комментарий.**
>
> - **Левый график** показывает асимметричное распределение активности.
>   Поэтому для описания такой переменной полезно смотреть не только среднее,
>   но и медиану, квартили и другие перцентили.
> - **Правый график** визуально подтверждает результат предыдущей ячейки:
>   доля успешного завершения составляет примерно **65% без сигнала**
>   и **36% в группе сигнала низкой активности**.
> - Это сравнение относится к **группам**, а не к конкретному человеку.
>   В группе сигнала есть обучающиеся, которые успешно завершили курс,
>   а среди остальных есть `Fail` и `Withdrawn`.
> - График не доказывает причинность и не является основанием для автоматического решения.
>   Его задача — помочь педагогу увидеть статистический сигнал и сформулировать вопрос для проверки.


---
## Ячейка 5. SQL-аналитика в BigQuery Sandbox

В примере преподавателя уже существует проект **`quality-management-analysis`**.
Если у студента проект уже есть, создавать новый не нужно. Если проекта нет,
его можно создать отдельно и затем использовать его **Project ID** в SQL.

### Порядок действий в текущем интерфейсе BigQuery

1. Откройте `https://console.cloud.google.com/bigquery` и убедитесь, что сверху отображается **Sandbox**.
2. В панели **Explorer** раскройте свой проект.
   В примере преподавателя: `quality-management-analysis`.
3. Нажмите **Datasets** → **Create dataset**.
   Если кнопка не отображается: `⋮` рядом с проектом → **Create dataset**.
4. Укажите:
   - **Dataset ID:** `learning_analytics`;
   - **Location type:** `Multi-region`;
   - **Location:** `EU`.
5. Откройте `learning_analytics` → **Create table**.
6. В разделе **Source**:
   - **Create table from:** `Upload`;
   - файл: `practice01_oulad.csv` (**в нашем решении 6,66 МБ**);
   - **File format:** `CSV`.
7. В разделе **Destination**:
   - Dataset: `learning_analytics`;
   - Table: `oulad_events`;
   - Table type: `Native table`.
8. В разделе **Schema** включите **Auto detect**.
   В **Advanced options** задайте **Header rows to skip = 1**.
9. После создания откройте:
   - **Schema** — `id_student`, `day`, `sum_click`, `id_site` должны быть `INTEGER`;
   - **Preview** — проверьте первые строки.
10. Для SQL нажмите **SQL query** либо у таблицы выберите **Query → In new tab**.

> **Важно:** в SQL используется именно **Project ID**.
> Если ваш Project ID отличается от `quality-management-analysis`,
> замените его во всех запросах ниже.

> **Sandbox:** не используем `INSERT`, `UPDATE`, `DELETE`, `MERGE` и streaming.
> Таблицы в песочнице имеют ограниченный срок жизни, поэтому практику строим на `SELECT`.


### Проверка 0. Убедимся, что в BigQuery загружен тот же учебный файл

```sql
SELECT
  COUNT(*) AS rows_count,
  COUNT(DISTINCT id_student) AS students,
  MIN(day) AS min_day,
  MAX(day) AS max_day
FROM `quality-management-analysis.learning_analytics.oulad_events`;
```

**Ожидаемый результат для эталонного файла:**

- `rows_count = 161160`;
- `students = 1819`;
- `min_day = 0`;
- `max_day = 59`.

Если значения другие, дальнейшие результаты с Colab совпадать не будут.

### Запрос 1. Базовые агрегаты за первые 30 дней — аналог `groupby` в pandas

```sql
SELECT
  id_student,
  SUM(sum_click)          AS total_clicks,
  COUNT(DISTINCT day)     AS active_days,
  COUNT(DISTINCT id_site) AS resources_used
FROM `quality-management-analysis.learning_analytics.oulad_events`
WHERE day BETWEEN 0 AND 29
GROUP BY id_student
ORDER BY total_clicks DESC;
```

Для эталонного файла запрос возвращает **1 767 строк** — по одной строке
на каждого обучающегося, у которого есть активность в первые 30 дней.

### Запрос 2. Оконная функция: накопленная активность

```sql
WITH daily_activity AS (
  SELECT
    id_student,
    day,
    SUM(sum_click) AS daily_clicks
  FROM `quality-management-analysis.learning_analytics.oulad_events`
  WHERE day BETWEEN 0 AND 29
  GROUP BY id_student, day
)

SELECT
  id_student,
  day,
  daily_clicks,
  SUM(daily_clicks) OVER (
    PARTITION BY id_student
    ORDER BY day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS cumulative_clicks
FROM daily_activity
ORDER BY id_student, day;
```

### Запрос 3. `LAG`: паузы между активными днями

```sql
WITH active_days_list AS (
  SELECT DISTINCT
    id_student,
    day
  FROM `quality-management-analysis.learning_analytics.oulad_events`
  WHERE day BETWEEN 0 AND 29
),

with_previous AS (
  SELECT
    id_student,
    day,
    LAG(day) OVER (
      PARTITION BY id_student
      ORDER BY day
    ) AS prev_active_day
  FROM active_days_list
)

SELECT
  id_student,
  day,
  prev_active_day,
  day - prev_active_day AS days_inactive_gap
FROM with_previous
WHERE day - prev_active_day >= 7
ORDER BY days_inactive_gap DESC;
```

Здесь пауза — **описательный показатель**. В этой практической мы не доказываем,
что она предсказывает итог лучше числа кликов.

### Запрос 4. Полное воспроизведение правила из pandas в SQL

```sql
WITH metrics AS (
  SELECT
    id_student,
    ANY_VALUE(final_result) AS final_result,
    SUM(sum_click)          AS total_clicks,
    COUNT(DISTINCT day)     AS active_days
  FROM `quality-management-analysis.learning_analytics.oulad_events`
  WHERE day BETWEEN 0 AND 29
  GROUP BY id_student
),

thresholds AS (
  SELECT
    PERCENTILE_CONT(total_clicks, 0.25) OVER () AS q25_clicks,
    PERCENTILE_CONT(active_days, 0.25) OVER () AS q25_days
  FROM metrics
  LIMIT 1
),

flagged AS (
  SELECT
    m.*,
    t.q25_clicks,
    t.q25_days,
    (
      m.total_clicks <= t.q25_clicks
      AND m.active_days <= t.q25_days
    ) AS support_signal
  FROM metrics AS m
  CROSS JOIN thresholds AS t
)

SELECT
  support_signal,
  COUNT(*) AS students_count,
  ROUND(
    AVG(IF(final_result IN ('Pass', 'Distinction'), 1.0, 0.0)),
    3
  ) AS success_rate,
  ANY_VALUE(q25_clicks) AS q25_clicks,
  ANY_VALUE(q25_days) AS q25_days
FROM flagged
GROUP BY support_signal
ORDER BY support_signal;
```

**Ожидаемый результат для эталонного файла:**

| support_signal | students_count | success_rate | q25_clicks | q25_days |
|---|---:|---:|---:|---:|
| `FALSE` | 1380 | 0.653 | 34 | 4 |
| `TRUE`  | 387  | 0.364 | 34 | 4 |

> **Методический комментарий.**
>
> - Сначала всегда выполняйте **Проверку 0**. Она подтверждает, что Colab и BigQuery
>   работают с одним и тем же файлом.
> - `GROUP BY` преобразует подробные записи в показатели на уровне обучающегося.
>   Это прямой аналог `pandas.groupby(...)`.
> - `SUM() OVER (...)` сохраняет последовательность по дням и показывает накопление активности.
> - `LAG()` нужен для описания интервалов между соседними активными днями.
>   В этой работе не называем такие интервалы причиной `Withdrawn`.
> - Запрос 4 теперь использует **два порога одновременно** — и по кликам, и по активным дням,
>   поэтому его результат соответствует Python-ячейке 3.
> - Если BigQuery определил `day` или `sum_click` как `STRING`, пересоздайте таблицу
>   с корректной схемой. Для эталонного CSV пропусков в этих полях нет, ожидаемый тип — `INTEGER`.


---
## Ячейка 6. Сверка результатов Python и BigQuery (контроль воспроизводимости)

Студент сохраняет результат Запроса 1 (**Save results → CSV**) и загружает файл обратно в Colab.

In [ ]:
BQ_RESULT = Path("bq_query1_result.csv")

if BQ_RESULT.exists():
    bq_df = pd.read_csv(BQ_RESULT)

    py_check = (
        student_metrics[
            ["id_student", "total_clicks", "active_days", "resources_used"]
        ]
        .sort_values("id_student")
        .reset_index(drop=True)
    )

    sql_check = (
        bq_df[
            ["id_student", "total_clicks", "active_days", "resources_used"]
        ]
        .sort_values("id_student")
        .reset_index(drop=True)
    )

    print(f"Строк в pandas: {len(py_check):,}")
    print(f"Строк в BigQuery: {len(sql_check):,}")
    print("Для эталонного среза ожидается 1 767 строк.")

    try:
        pd.testing.assert_frame_equal(
            py_check,
            sql_check,
            check_dtype=False
        )
        print(
            "Контроль пройден: pandas и BigQuery дали одинаковые "
            "id_student, total_clicks, active_days и resources_used."
        )
    except AssertionError as e:
        print("Обнаружено расхождение.")
        print(
            "Проверьте: тот ли CSV загружен, одинаков ли период 0–29, "
            "и верно ли определены числовые типы."
        )
        print(str(e)[:1200])
else:
    print(
        "Файл bq_query1_result.csv не найден. "
        "Сначала выполните Запрос 1 в BigQuery и сохраните результат в CSV."
    )


> **Методический комментарий.**
>
> - Для эталонного набора обе среды должны вернуть **1 767 строк** ранних агрегатов.
> - Мы проверяем сразу три показателя: `total_clicks`, `active_days`, `resources_used`.
>   Если они совпали для всех `id_student`, вычислительная логика воспроизводима.
> - Если число строк отличается, сначала проверьте **Проверку 0** в BigQuery:
>   должно быть 161 160 строк, 1 819 обучающихся и дни 0–59.
> - Частые причины расхождений: другой файл; `day BETWEEN 0 AND 30` вместо `0 AND 29`;
>   неправильный тип `day`/`sum_click`; случайная фильтрация таблицы перед экспортом.
> - Смысл шага не в том, что SQL «правильнее» Python. Мы показываем,
>   что одна и та же аналитическая операция воспроизводится в двух средах.


---
## Ячейка 7. Педагогическая интерпретация и вывод (образец ответа)

> **Образец заключения слушателя-педагога**
>
> «В ходе анализа ранней активности на курсе `BBB`, поток `2013J`,
> за первые 30 дней в расчёт вошли **1 767 обучающихся**.
> Учебное правило выделило **387 человек**, у которых одновременно
> суммарная активность не превышала **34 взаимодействия**, а число активных дней — **4**.
>
> В этой группе доля итоговых результатов `Pass/Distinction` составила **36,4%**,
> тогда как среди остальных обучающихся — **65,3%**.
> Это показывает статистическую связь раннего цифрового следа
> с последующим результатом в данном конкретном потоке, но не доказывает причинность.
>
> Индикатор нельзя трактовать как оценку способностей, мотивации или отношения к предмету.
> Низкая активность в LMS может быть связана с доступом к сети, использованием материалов офлайн,
> особенностями интерфейса, состоянием здоровья, занятостью или иной стратегией обучения.
>
> Этически корректное действие после завершения 30-дневного окна —
> использовать сигнал как повод для доброжелательной педагогической проверки:
> предложить консультацию и уточнить, нужна ли организационная или методическая помощь.
> Если поддержку необходимо организовать уже на 15-й день, пороги нужно **пересчитать
> по данным дней 0–14**, а не применять значения, полученные для 30-дневного окна.
>
> Недопустимо автоматически снижать баллы, маркировать человека как „неуспевающего“
> или публиковать списки обучающихся на основе одного индикатора».

> **Методический комментарий.**
>
> - В итоговом ответе должны присутствовать **конкретные числа**, которые действительно получены:
>   `34`, `4`, `387/1767`, `36,4%` и `65,3%`.
> - Формулировка должна сохранять различие между **ассоциацией** и **причинностью**.
> - Педагогическое действие должно соответствовать временному окну анализа:
>   30-дневный индикатор нельзя честно использовать «на 15-й день» без перерасчёта.
> - В нашем учебном файле сознательно отсутствуют ФИО, контакты, геолокация, изображения
>   и иные данные, не нужные для поставленной аналитической задачи.
> - Сигнал предназначен для **проверки и поддержки**, а не для санкций или автоматического решения.

---

### Приложение. Разбор нестандартных вариантов

| Вариант | На что обратить внимание при проверке |
|:---:|---|
| 3, 12 | Разрыв считается только между **соседними активными** днями; предварительный `DISTINCT` обязателен |
| 5, 6 | HarvardX — уровень «студент × курс», агрегация по событиям не нужна; обязателен `PARTITION BY course_id` |
| 8 | Демография — только для отдельного аудита справедливости, а не для автоматического педагогического решения |
| 9, 11 | Метки времени SNAP — относительные секунды; перевод в дни нужно проговаривать явно |
| 13–16 | Duolingo: большой событийный набор, срез следует формировать по целым пользователям |
| 15 | Деление `history_correct / history_seen` выполняется только с защитой от деления на ноль |
| 17, 20 | Junyi: если Kaggle недоступен, преподаватель заранее выдаёт подготовленный локальный срез |
| 11, 20 | При работе с оконными рамками важно помнить, включает ли окно текущую строку |
